In [ ]:
library(ggplot2)
library(data.table)
library(tidyverse)
library(forcats)
library(biomaRt)
library(scales)

In [ ]:
system("wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.annotation.gff3.gz",
      intern=TRUE)

In [ ]:
system("zgrep 'ENSG00000119772' gencode.v49.annotation.gff3.gz > gencode.ENSG00000119772.gff3")
system("zgrep 'ENSG00000161547' gencode.v49.annotation.gff3.gz > gencode.ENSG00000161547.gff3")
system("zgrep 'ENSG00000182054' gencode.v49.annotation.gff3.gz > gencode.ENSG00000182054.gff3")
system("zgrep 'ENSG00000168769' gencode.v49.annotation.gff3.gz > gencode.ENSG00000168769.gff3")

In [ ]:
system("wc -l clinvar.vcf", intern=TRUE)

In [ ]:
single_assoc <- fread("EUR-AML-Rare-Variants-Deprivation-Index.singleAssoc.txt")
ultra_rare_marker_list <- fread("EUR-AML-Ultra-Rare-Variants-Deprivation-Index.markerList.txt")
rare_marker_list <- fread("EUR-AML-Rare-Variants-Deprivation-Index.markerList.txt")

In [ ]:
genes <- c("ENSG00000119772", "ENSG00000161547", "ENSG00000182054", "ENSG00000168769")

In [ ]:
mart <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")

In [ ]:
options(repr.plot.width = 20, height = 8)

In [ ]:
for (gene in genes) {
  genomic_region <- fread(paste0("gencode.", gene, ".gff3")) %>%
    filter(V3 %in% c("exon", "gene")) %>%
    mutate(V10 = 1, V11 = -1,
           across(c(V4, V5, V10, V11), as.numeric)) %>%
    filter(!is.na(V4) & !is.na(V5))
  
  ultra_rare_gene_list <- ultra_rare_marker_list %>%
    filter(Region == gene) %>%
    separate(Ultra_Rare_Variants, into = c("CHR", "POS", "REF", "ALT"), sep = ":") %>%
    mutate(POS = suppressWarnings(as.numeric(trimws(POS)))) %>%
    filter(!is.na(POS))
  
  rare_gene_list <- rare_marker_list %>%
    filter(Region == gene) %>%
    separate(Rare_Variants, into = c("CHR", "POS", "REF", "ALT"), sep = ":") %>%
    mutate(POS = suppressWarnings(as.numeric(trimws(POS)))) %>%
    filter(!is.na(POS))
  
  my_colors <- c("exon" = "#088BBEFF", "gene" = "#F9F9F9FF")
    
  gene_map <- getBM(
      attributes = c("ensembl_gene_id", "hgnc_symbol"),
      filters = "ensembl_gene_id",
      values = gene,
      mart = mart
    )
  
  plot <- ggplot() +
    # Ultra-rare variants
    geom_segment(data = ultra_rare_gene_list,
                 aes(x = POS, xend = POS, y = 1, yend = 3),
                 linewidth = 1, color = "black") +
    geom_point(data = ultra_rare_gene_list,
               aes(x = POS, y = 3), color = "#F7AA14FF", size = 3) +
    # Rare variants
    geom_segment(data = rare_gene_list,
                 aes(x = POS, xend = POS, y = -1, yend = -3),
                 linewidth = 1, color = "black") +
    geom_point(data = rare_gene_list,
               aes(x = POS, y = -3), color = "#F5D000FF", size = 3) +
    # Gene features
    geom_rect(data = genomic_region,
              aes(xmin = V4, xmax = V5, ymin = V11, ymax = V10, fill = V3),
              color = "black", alpha = 0.3) +
    scale_fill_manual(values = my_colors) +
    coord_cartesian(ylim = c(-6, 6)) +
    theme_minimal() +
    theme(
    axis.text.y = element_blank(),  
    axis.ticks.y = element_blank()) + 
    labs(title = paste("Gene:", gene_map$hgnc_symbol),
         x = "Genomic position",
         y = "",
         color = "Region") + 
    guides(fill = "none")
  # Display plot 
  print(plot)
  # Save PNG
  ggsave(paste0(gene, "_gene_plot.png"),
         plot = plot, height = 6, width = 20, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0(gene, "_gene_plot.rds"))

}


In [ ]:
gdc_frequent_mutations <- fread("GDC-frequent-mutations.tsv")

ultra_rare_hgvs <- fread("EUR-AML-Ultra-Rare-HGVS-Deprivation-Index.markerList.txt")
rare_hgvs <- fread("EUR-AML-Rare-HGVS-Deprivation-Index.markerList.txt")

ultra_rare_clinvar <- fread("EUR-AML-Ultra-Rare-Clinvar-Deprivation-Index.markerList.txt")
rare_marker_clinvar <- fread("EUR-AML-Rare-Clinvar-Deprivation-Index.markerList.txt")

group_annotation <- fread("Group-Annotation.txt")

In [ ]:
gdc_frequent_mutations <- 
    gdc_frequent_mutations %>% 
    rename("HGVS" = "dna_change",
           "GDC_Ultra_Rare_Variants" = "AoU_Ultra_Rare_Variants", 
           "GDC_Rare_Variants" = "AoU_Rare_Variants") %>% 
    dplyr::select(HGVS, GDC_Ultra_Rare_Variants, GDC_Rare_Variants) %>% 
    mutate(GDC_Ultra_Rare_Variants = ifelse(GDC_Ultra_Rare_Variants == "Match", 1, 0),
           GDC_Rare_Variants = ifelse(GDC_Rare_Variants == "Match", 1, 0))

In [ ]:
ultra_rare_merge <- 
ultra_rare_clinvar %>% 
    mutate(Clinvar_Ultra_Rare_Variants = 1) %>% 
     dplyr::select(Variants, Clinvar_Ultra_Rare_Variants) %>% 
    rename("Ultra_Rare_Variants" = "Variants") %>% 
    full_join(ultra_rare_hgvs, by = "Ultra_Rare_Variants") %>% 
    left_join(gdc_frequent_mutations, by = "HGVS") 
ultra_rare_merge[is.na(ultra_rare_merge)] <- 0 

rare_merge <- 
rare_marker_clinvar %>% 
    mutate(Clinvar_Rare_Variants = 1) %>% 
    dplyr::select(Variants, Clinvar_Rare_Variants) %>% 
    rename("Rare_Variants" = "Variants") %>% 
    full_join(rare_hgvs, by = "Rare_Variants") %>% 
    left_join(gdc_frequent_mutations, by = "HGVS") 
rare_merge[is.na(rare_merge)] <- 0 

In [ ]:
ultra_rare_merge <- 
ultra_rare_merge %>% 
    mutate(GDC_Clinvar_Ultra_Rare_Variants = ifelse(GDC_Ultra_Rare_Variants == 1 | 
                                                    Clinvar_Ultra_Rare_Variants == 1, 1, 0)) %>% 
    group_by(Region) %>% 
    summarize(GDC_Percentage = mean(GDC_Ultra_Rare_Variants),
              Clinvar_Percentage = mean(Clinvar_Ultra_Rare_Variants),
              GDC_Clinvar_Percentage = mean(GDC_Clinvar_Ultra_Rare_Variants)) %>% 
    pivot_longer(
        cols = -c(Region),  
        names_to = "Type",          
        values_to = "Match") %>% 
    mutate(No_Match = 1 - Match) %>% 
    pivot_longer(
        cols = -c(Region, Type),  
        names_to = "Match",          
        values_to = "Percentage") %>% 
    mutate(Match = gsub("_", " ", Match)) %>% 
    mutate(Type = gsub("_Percentage", " ", Type)) %>% 
    mutate(Type = gsub("_", " & ", Type))

gene_map <- getBM(
      attributes = c("ensembl_gene_id", "hgnc_symbol"),
      filters = "ensembl_gene_id",
      values = ultra_rare_merge$Region,
      mart = mart
    )
colnames(gene_map)[1] <- "Region"
ultra_rare_merge <- inner_join(ultra_rare_merge, gene_map, by = "Region")

rare_merge <- 
rare_merge %>% 
    mutate(GDC_Clinvar_Rare_Variants = ifelse(GDC_Rare_Variants == 1 | 
                                                    Clinvar_Rare_Variants == 1, 1, 0)) %>% 
    group_by(Region) %>% 
    summarize(GDC_Percentage = mean(GDC_Rare_Variants),
              Clinvar_Percentage = mean(Clinvar_Rare_Variants),
              GDC_Clinvar_Percentage = mean(GDC_Clinvar_Rare_Variants)) %>% 
    pivot_longer(
        cols = -c(Region),  
        names_to = "Type",          
        values_to = "Match") %>% 
    mutate(No_Match = 1 - Match) %>% 
    pivot_longer(
        cols = -c(Region, Type),  
        names_to = "Match",          
        values_to = "Percentage") %>% 
    mutate(Match = gsub("_", " ", Match)) %>% 
    mutate(Type = gsub("_Percentage", " ", Type)) %>% 
    mutate(Type = gsub("_", " & ", Type))

gene_map <- getBM(
      attributes = c("ensembl_gene_id", "hgnc_symbol"),
      filters = "ensembl_gene_id",
      values = rare_merge$Region,
      mart = mart
    )
colnames(gene_map)[1] <- "Region"
rare_merge <- inner_join(rare_merge, gene_map, by = "Region")



In [ ]:
options(repr.plot.width = 8, height = 8)

In [ ]:
# Get the colors from the palette
all_colors <- paletteer::paletteer_d("LaCroixColoR::Lemon")
selected_colors <- all_colors[c(6, 2)]  # pick first and sixth

plot <- 
ggplot(ultra_rare_merge, 
       aes(x = Type, y = Percentage, fill = fct_rev(Match))) +
  geom_bar(position = "fill", stat = "identity") +
  scale_y_continuous(labels = percent_format()) +
  facet_wrap(vars(hgnc_symbol)) +
  theme_minimal() +
  scale_fill_manual(values = selected_colors) +
  labs(fill = "Status") +  # set legend title
  theme(
    plot.title = element_text(size = 20, face = "bold", hjust = 0.5), # Increase size, center title
    strip.text = element_text(size = 14),   # Facet labels
    axis.title = element_text(size = 14),   # Axis titles
    axis.text = element_text(size = 12),
    axis.text.x = element_text(angle = 15)  # Axis tick labels
  )

  # Save PNG
  ggsave(paste0("ultra_rare_percentage_plot.png"),
         plot = plot, height = 8, width = 8, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0("ultra_rare_percentage_plot.rds"))



In [ ]:
# Get the colors from the palette
all_colors <- paletteer::paletteer_d("LaCroixColoR::Lemon")
selected_colors <- all_colors[c(6, 2)]  # pick first and sixth

plot <- 
ggplot(rare_merge, 
       aes(x = Type, y = Percentage, fill = fct_rev(Match))) +
  geom_bar(position = "fill", stat = "identity") +
  scale_y_continuous(labels = percent_format()) +
  facet_wrap(vars(hgnc_symbol)) +
  theme_minimal() +
  scale_fill_manual(values = selected_colors) +
  labs(fill = "Status")  +  # set legend title
  theme(
    plot.title = element_text(size = 20, face = "bold", hjust = 0.5), # Increase size, center title
    strip.text = element_text(size = 14),   # Facet labels
    axis.title = element_text(size = 14),   # Axis titles
    axis.text = element_text(size = 12),
    axis.text.x = element_text(angle = 15)  # Axis tick labels
  )

  # Save PNG
  ggsave(paste0("rare_percentage_plot.png"),
         plot = plot, height = 8, width = 8, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0("rare_percentage_plot.rds"))



In [ ]:
gdc_frequent_mutations <- fread("GDC-frequent-mutations.tsv")

ultra_rare_hgvs <- fread("EUR-AML-Ultra-Rare-HGVS-Deprivation-Index.markerList.txt")
rare_hgvs <- fread("EUR-AML-Rare-HGVS-Deprivation-Index.markerList.txt")

ultra_rare_clinvar <- fread("EUR-AML-Ultra-Rare-Clinvar-Deprivation-Index.markerList.txt")
rare_marker_clinvar <- fread("EUR-AML-Rare-Clinvar-Deprivation-Index.markerList.txt")

group_annotation <- fread("Group-Annotation.txt")

In [ ]:
gdc_frequent_mutations <- 
    gdc_frequent_mutations %>% 
    rename("HGVS" = "dna_change",
           "GDC_Ultra_Rare_Variants" = "AoU_Ultra_Rare_Variants", 
           "GDC_Rare_Variants" = "AoU_Rare_Variants") %>% 
    dplyr::select(HGVS, GDC_Ultra_Rare_Variants, GDC_Rare_Variants) %>% 
    mutate(GDC_Ultra_Rare_Variants = ifelse(GDC_Ultra_Rare_Variants == "Match", 1, 0),
           GDC_Rare_Variants = ifelse(GDC_Rare_Variants == "Match", 1, 0))

In [ ]:
ultra_rare_merge <- 
ultra_rare_clinvar %>% 
    mutate(Clinvar_Ultra_Rare_Variants = 1) %>% 
     dplyr::select(Variants, Clinvar_Ultra_Rare_Variants) %>% 
    rename("Ultra_Rare_Variants" = "Variants") %>% 
    full_join(ultra_rare_hgvs, by = "Ultra_Rare_Variants") %>% 
    left_join(gdc_frequent_mutations, by = "HGVS") %>% 
    inner_join(rename(group_annotation, "Ultra_Rare_Variants" = "var"), 
               by = c("Ultra_Rare_Variants", "Region"))
ultra_rare_merge[is.na(ultra_rare_merge)] <- 0 

rare_merge <- 
rare_marker_clinvar %>% 
    mutate(Clinvar_Rare_Variants = 1) %>% 
    dplyr::select(Variants, Clinvar_Rare_Variants) %>% 
    rename("Rare_Variants" = "Variants") %>% 
    full_join(rare_hgvs, by = "Rare_Variants") %>% 
    left_join(gdc_frequent_mutations, by = "HGVS") %>% 
    inner_join(rename(group_annotation, "Rare_Variants" = "var"), 
               by = c("Rare_Variants", "Region"))
rare_merge[is.na(rare_merge)] <- 0 

In [ ]:
ultra_rare_merge <- 
ultra_rare_merge %>% 
    mutate(GDC_Clinvar_Ultra_Rare_Variants = ifelse(GDC_Ultra_Rare_Variants == 1 | 
                                                    Clinvar_Ultra_Rare_Variants == 1, 1, 0)) %>% 
    group_by(Region, anno) %>% 
    summarize(GDC_Percentage = mean(GDC_Ultra_Rare_Variants),
              Clinvar_Percentage = mean(Clinvar_Ultra_Rare_Variants),
              GDC_Clinvar_Percentage = mean(GDC_Clinvar_Ultra_Rare_Variants)) %>% 
    pivot_longer(
        cols = -c(Region, anno),  
        names_to = "Type",          
        values_to = "Match") %>% 
    mutate(No_Match = 1 - Match) %>% 
    pivot_longer(
        cols = -c(Region, Type, anno),  
        names_to = "Match",          
        values_to = "Percentage") %>% 
    mutate(Match = gsub("_", " ", Match)) %>% 
    mutate(Type = gsub("_Percentage", " ", Type)) %>% 
    mutate(Type = gsub("_", " & ", Type))

gene_map <- getBM(
      attributes = c("ensembl_gene_id", "hgnc_symbol"),
      filters = "ensembl_gene_id",
      values = ultra_rare_merge$Region,
      mart = mart
    )
colnames(gene_map)[1] <- "Region"
ultra_rare_merge <- inner_join(ultra_rare_merge, gene_map, by = "Region")

rare_merge <- 
rare_merge %>% 
    mutate(GDC_Clinvar_Rare_Variants = ifelse(GDC_Rare_Variants == 1 | 
                                                    Clinvar_Rare_Variants == 1, 1, 0)) %>% 
    group_by(Region, anno) %>% 
    summarize(GDC_Percentage = mean(GDC_Rare_Variants),
              Clinvar_Percentage = mean(Clinvar_Rare_Variants),
              GDC_Clinvar_Percentage = mean(GDC_Clinvar_Rare_Variants)) %>% 
    pivot_longer(
        cols = -c(Region, anno),  
        names_to = "Type",          
        values_to = "Match") %>% 
    mutate(No_Match = 1 - Match) %>% 
    pivot_longer(
        cols = -c(Region, Type, anno),  
        names_to = "Match",          
        values_to = "Percentage") %>% 
    mutate(Match = gsub("_", " ", Match)) %>% 
    mutate(Type = gsub("_Percentage", " ", Type)) %>% 
    mutate(Type = gsub("_", " & ", Type))

gene_map <- getBM(
      attributes = c("ensembl_gene_id", "hgnc_symbol"),
      filters = "ensembl_gene_id",
      values = rare_merge$Region,
      mart = mart
    )
colnames(gene_map)[1] <- "Region"
rare_merge <- inner_join(rare_merge, gene_map, by = "Region")



In [ ]:
options(repr.plot.width = 12, height = 16)

In [ ]:
# Get the colors from the palette
all_colors <- paletteer::paletteer_d("LaCroixColoR::Lemon")
selected_colors <- all_colors[c(6, 2)]  # pick first and sixth

plot <- 
ggplot(ultra_rare_merge, 
       aes(x = Type, y = Percentage, fill = fct_rev(Match))) +
  geom_bar(position = "fill", stat = "identity") +
  scale_y_continuous(labels = percent_format()) +
  facet_wrap(vars(hgnc_symbol, anno), ncol = 3) +
  theme_minimal() +
  scale_fill_manual(values = selected_colors) +
  labs(fill = "Status")  +  # set legend title
  theme(
    plot.title = element_text(size = 20, face = "bold", hjust = 0.5), # Increase size, center title
    strip.text = element_text(size = 14),   # Facet labels
    axis.title = element_text(size = 14),   # Axis titles
    axis.text = element_text(size = 12),
    axis.text.x = element_text(angle = 15)  # Axis tick labels
  )

  # Save PNG
  ggsave(paste0("ultra_rare_annotation_percentage_plot.png"),
         plot = plot, height = 16, width = 16, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0("ultra_rare_annotation_percentage_plot.rds"))



In [ ]:
options(repr.plot.width = 16, height = 8)

In [ ]:
# Get the colors from the palette
all_colors <- paletteer::paletteer_d("LaCroixColoR::Lemon")
selected_colors <- all_colors[c(6, 2)]  # pick first and sixth

plot <-
ggplot(rare_merge, 
       aes(x = Type, y = Percentage, fill = fct_rev(Match))) +
  geom_bar(position = "fill", stat = "identity") +
  scale_y_continuous(labels = percent_format()) +
  facet_wrap(vars(hgnc_symbol, anno), ncol = 4) +
  theme_minimal() +
  scale_fill_manual(values = selected_colors) +
  labs(fill = "Status") +  # set legend title
  theme(
    plot.title = element_text(size = 20, face = "bold", hjust = 0.5), # Increase size, center title
    strip.text = element_text(size = 14),   # Facet labels
    axis.title = element_text(size = 14),   # Axis titles
    axis.text = element_text(size = 12),
    axis.text.x = element_text(angle = 15)  # Axis tick labels
  )

  # Save PNG
  ggsave(paste0("rare_annotation_percentage_plot.png"),
         plot = plot, height = 8, width = 16, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0("rare_annotation_percentage_plot.rds"))



In [ ]:
options(repr.plot.width = 10, height = 8)

In [ ]:
# Split each row by "|", returns a list
CLNDN_list <- strsplit(ultra_rare_clinvar$CLNDN, "\\|")
# Flatten the list into a single vector
CLNDN <- unlist(CLNDN_list)
# Optional: remove leading/trailing whitespace
CLNDN <- trimws(CLNDN)
CLNDN <- as.data.frame(table(CLNDN))
CLNDN$Names <- gsub("_", " ", CLNDN$CLNDN)
CLNDN$Names  <- paste0(
  toupper(substring(CLNDN$Names , 1, 1)),  
  substring(CLNDN$Names , 2)              
)

plot <- 
ggplot(data = CLNDN) +
   geom_segment(aes(y = fct_reorder(Names, Freq), 
                    yend = fct_reorder(Names, Freq), 
                    x = 0, 
                    xend = Freq),
                 size = 3, color = "black") +
   geom_point(aes(y = fct_reorder(Names, Freq), 
                  x = Freq), color = "#F7AA14FF", size = 4) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(x = "Count", y = "Disorder", title = "Clinvar Phenotypes: Frequency of Disorders") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

  # Save PNG
  ggsave(paste0("ultra_rare_clinvar_phenotype_plot.png"),
         plot = plot, height = 8, width = 16, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0("ultra_rare_clinvar_phenotype_plot.rds"))


In [ ]:
# Split each row by "|", returns a list
CLNDN_list <- strsplit(rare_marker_clinvar$CLNDN, "\\|")
# Flatten the list into a single vector
CLNDN <- unlist(CLNDN_list)
# Optional: remove leading/trailing whitespace
CLNDN <- trimws(CLNDN)
CLNDN <- as.data.frame(table(CLNDN))
CLNDN$Names <- gsub("_", " ", CLNDN$CLNDN)
CLNDN$Names  <- paste0(
  toupper(substring(CLNDN$Names , 1, 1)),  
  substring(CLNDN$Names , 2)              
)

plot <- 
ggplot(data = CLNDN) +
   geom_segment(aes(y = fct_reorder(Names, Freq), 
                    yend = fct_reorder(Names, Freq), 
                    x = 0, 
                    xend = Freq),
                 size = 3, color = "black") +
   geom_point(aes(y = fct_reorder(Names, Freq), 
                  x = Freq), color = "#F5D000FF", size = 4) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(x = "Count", y = "Disorder", title = "Clinvar Phenotypes: Frequency of Disorders") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
  
  # Save PNG
  ggsave(paste0("rare_clinvar_phenotype_plot.png"),
         plot = plot, height = 8, width = 16, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0("rare_clinvar_phenotype_plot.rds"))



## EXTRA CODE 

In [ ]:
single_assoc %>% 
    mutate(MarkerID = as.factor(MarkerID)) %>% 
    mutate(Negative_Log = -log10(p.value.NA)) %>% 
    ggplot() + 
    geom_point(aes(x = fct_reorder(MarkerID, -Negative_Log), 
                   y = Negative_Log, 
                   size = Negative_Log)) + 
   geom_segment(aes(x = MarkerID, xend = MarkerID, y = 0, yend = Negative_Log),
               size = 1, color = "black") +
   theme_bw() + 
   theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
colnames(rare_marker_clinvar)[9] <- "MarkerID"

In [ ]:
inner_join(single_assoc, rare_marker_clinvar, by = "MarkerID") %>% dplyr::select(MarkerID, p.value.NA)

In [ ]:
gdc_frequent_mutations <- fread("GDC-frequent-mutations.tsv")

ultra_rare_hgvs <- fread("EUR-AML-Ultra-Rare-HGVS-Deprivation-Index.markerList.txt")
rare_hgvs <- fread("EUR-AML-Rare-HGVS-Deprivation-Index.markerList.txt")

ultra_rare_clinvar <- fread("EUR-AML-Ultra-Rare-Clinvar-Deprivation-Index.markerList.txt")
rare_marker_clinvar <- fread("EUR-AML-Rare-Clinvar-Deprivation-Index.markerList.txt")

In [ ]:
gdc_frequent_mutations[is.na(gdc_frequent_mutations)] <- 0 

In [ ]:
gdc_frequent_mutations <- 
    gdc_frequent_mutations %>% 
    mutate(total_num_ssm_affected_cases = pmax(num_DNMT3A_cases, num_TET2_cases, num_IDH2_cases, num_SRSF2_cases)) %>% 
    rename("HGVS" = "dna_change",
           "GDC_Ultra_Rare_Variants" = "AoU_Ultra_Rare_Variants", 
           "GDC_Rare_Variants" = "AoU_Rare_Variants") %>% 
    dplyr::select(HGVS, GDC_Ultra_Rare_Variants, GDC_Rare_Variants, num_ssm_affected_cases, total_num_ssm_affected_cases) %>% 
    mutate(GDC_Ultra_Rare_Variants = ifelse(GDC_Ultra_Rare_Variants == "Match", 1, 0),
           GDC_Rare_Variants = ifelse(GDC_Rare_Variants == "Match", 1, 0)) %>% 
    mutate(num_ssm_affected_cases_05 = (num_ssm_affected_cases/ total_num_ssm_affected_cases),
           num_ssm_affected_cases_05 = rescale(num_ssm_affected_cases_05, to = c(0, 5)))

In [ ]:
ultra_rare_merge <- 
ultra_rare_clinvar %>% 
    mutate(Clinvar_Ultra_Rare_Variants = 1) %>% 
    dplyr::select(Variants, Clinvar_Ultra_Rare_Variants) %>% 
    rename("Ultra_Rare_Variants" = "Variants") %>% 
    full_join(ultra_rare_hgvs, by = "Ultra_Rare_Variants") %>% 
    left_join(gdc_frequent_mutations, by = "HGVS") 


rare_merge <- 
rare_marker_clinvar %>% 
    mutate(Clinvar_Rare_Variants = 1) %>% 
    dplyr::select(Variants, Clinvar_Rare_Variants) %>% 
    rename("Rare_Variants" = "Variants") %>% 
    full_join(rare_hgvs, by = "Rare_Variants") %>% 
    left_join(gdc_frequent_mutations, by = "HGVS") 
rare_merge[is.na(rare_merge)] <- 0 

In [ ]:
for (gene in genes) {
  genomic_region <- fread(paste0("gencode.", gene, ".gff3")) %>%
    filter(V3 %in% c("exon", "gene")) %>%
    mutate(V10 = 1, V11 = -1,
           across(c(V4, V5, V10, V11), as.numeric)) %>%
    filter(!is.na(V4) & !is.na(V5))
  
  ultra_rare_gene_list <- ultra_rare_merge %>%
    filter(Region == gene) %>%
    filter(GDC_Ultra_Rare_Variants == 1) %>% 
    mutate(POS = suppressWarnings(as.numeric(trimws(POS)))) %>%
    filter(!is.na(POS))
  
  rare_gene_list <- rare_merge %>%
    filter(Region == gene) %>%
    filter(GDC_Rare_Variants == 1) %>% 
    mutate(POS = suppressWarnings(as.numeric(trimws(POS)))) %>%
    filter(!is.na(POS))
  
  my_colors <- c("exon" = "#088BBEFF", "gene" = "#F9F9F9FF")
  
  plot <- ggplot() +
    # Ultra-rare variants
    geom_segment(data = ultra_rare_gene_list,
                 aes(x = POS, xend = POS, y = 1, yend = num_ssm_affected_cases_05 + 1),
                 size = 1, color = "black") +
    geom_point(data = ultra_rare_gene_list,
               aes(x = POS, y = num_ssm_affected_cases_05 + 1), color = "#F7AA14FF", size = 3) +
    # Rare variants
    geom_segment(data = rare_gene_list,
                 aes(x = POS, xend = POS, y = -1, yend = -num_ssm_affected_cases_05 - 1),
                 size = 1, color = "black") +
    geom_point(data = rare_gene_list,
               aes(x = POS, y = -num_ssm_affected_cases_05 - 1), color = "#F5D000FF", size = 3) +
    # Gene features
    geom_rect(data = genomic_region,
              aes(xmin = V4, xmax = V5, ymin = V11, ymax = V10, fill = V3),
              color = "black", alpha = 0.3) +
    scale_fill_manual(values = my_colors) +
    coord_cartesian(ylim = c(-6, 6)) +
    theme_minimal() +
    theme(
    axis.text.y = element_blank(),  
    axis.ticks.y = element_blank()) + 
    labs(title = paste("Gene:", gene),
         x = "Genomic position",
         y = "",
         color = "Region") + 
    guides(fill = "none")
  # Display plot 
  print(plot)
  # Save PNG
  ggsave(paste0(gene, "_gene_plot.png"),
         plot = plot, height = 6, width = 20, dpi = 300)
  # Save RDS objects
  saveRDS(plot, paste0(gene, "_gene_plot.rds"))

}
